# Final Cross-Year Historical Integrity Validation
資料源驗證，沒有策略或returns。30 planned windows；as-of eligibility決定included窗口與request count。
先freeze官方矩陣與pytest；future窗口保留但NOT_YET_ELIGIBLE，不替換、不作data quality判斷。前階段baseline d5b7cf2。
External官方資料不可取得時保留UNRESOLVED；NOT COMPARABLE不可假稱external PASS。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from google.colab import userdata
REPO_URL = "https://github.com/hh4832/fubon_futures_intraday_data_validation.git"
REPO_DIR = Path('/content/fubon_futures_intraday_data_validation_final')
BRANCH = 'feature/final-cross-year-historical-integrity-validation'
# Public clone first; only a private repo needs optional GITHUB_TOKEN.
if not REPO_DIR.exists():
    result = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],capture_output=True,text=True)
    if result.returncode:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            raise RuntimeError('Public clone failed; private repositories need GITHUB_TOKEN') from None
        if not token:
            raise RuntimeError('GITHUB_TOKEN unavailable')
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp)/'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["COLAB_CLONE_TOKEN"])\n')
            askpass.chmod(0o700)
            env = dict(os.environ,GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',COLAB_CLONE_TOKEN=token)
            retry = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],env=env,capture_output=True)
            del env,token
            if retry.returncode:
                raise RuntimeError('Authenticated clone failed; output suppressed')
remote = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
if remote != REPO_URL:
    raise RuntimeError('Repository origin differs from expected URL')
os.chdir(REPO_DIR)
sys.path.insert(0,str(REPO_DIR))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)

actual_branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO_DIR,text=True).strip()
if actual_branch != BRANCH:
    raise RuntimeError('Use the rollover feature branch checkout; no automatic branch replacement')

In [ ]:
from src.archive import git_info
provenance = git_info(REPO_DIR)
print(provenance)
if provenance['git_status']:
    raise RuntimeError('Use a clean Git checkout; no auto pull or overwrite')

In [ ]:
import platform
print({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'architecture':platform.machine()})
from src.config import load_config
cfg = load_config()
from src.final_sample_matrix import load_final_config
cfg = load_final_config()
print({k:cfg[k] for k in ["product","sessions","timeframe","continuous_contract"]})
display(cfg["samples"])

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from scripts.freeze_final_samples import freeze
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)
preflight_dir = Path(cfg["drive_root"]) / (datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d_%H%M%S") + "_" + provenance["git_commit"][:12] + "_final_preflight")
matrix, preflight_info = freeze(preflight_dir)
print(preflight_info)
matrix.require_completed()  # Checks included sessions only; READY is not live authorization.
from scripts.replay_final_validator_corrections import replay_archive
source_archive = Path(cfg["drive_root"]) / cfg["validator_correction_provenance"]["previous_formal_run_id"]
replay_dir, replay_info = replay_archive(source_archive, preflight_dir.with_name(preflight_dir.name + "_validator_replay"))
print({key:replay_info[key] for key in ["replay_status", "known_2018_external_case", "api_request_count", "formal_rerun_required"]})
if replay_info["replay_status"] != "PASS":
    raise RuntimeError("Offline validator replay gate failed; no formal rerun.")
LIVE_FUBON_REQUESTS_APPROVED = False
if not LIVE_FUBON_REQUESTS_APPROVED:
    raise RuntimeError("Preflight complete; wait for explicit approval before SDK/auth or live market requests.")


In [ ]:
from tqdm.auto import tqdm
from scripts.install_fubon_sdk_colab import install
for _ in tqdm(range(1),desc='SDK setup'):
    sdk_info = install(cfg['sdk_version'],cfg['sdk_download_url'],cfg['sdk_official_sha256'])

In [ ]:
import fubon_neo
from fubon_neo.sdk import FubonSDK
print("SDK import PASS")

In [ ]:
from src.fubon_auth import load_secrets, validate_certificate, login
secrets = load_secrets(userdata.get)

In [ ]:
validate_certificate(secrets)

In [ ]:
sdk, auth_info = login(secrets)
del secrets

In [ ]:
try:
    sdk.init_realtime()
    restfutopt = sdk.marketdata.rest_client.futopt
except Exception:
    raise RuntimeError('Market-data initialization failed; SDK exception suppressed') from None
from src.historical_client import read_canonical_documentation
_, documentation_info = read_canonical_documentation(cfg)
print(documentation_info)


In [ ]:
from src.final_integrity_runner import run_final_integrity
run_dir, summary = run_final_integrity(restfutopt, REPO_DIR, sdk_info, auth_info, cfg, now=matrix.as_of)
print(summary)


In [ ]:
import pandas as pd
for name in ["cross_year_yearly_summary.csv", "final_historical_integrity_summary.csv", "external_taifex_spot_check.csv", "historical_availability_boundary.csv"]:
    display(pd.read_csv(run_dir / "diagnostics" / name))
